In [1]:
# ── Cell 1: Installs & Imports ───────────────────────────────
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
import pandas as pd
import os, time

SEEDS        = [42, 123, 456]
BUDGET       = 5000
RESULTS_DIR  = "/kaggle/working/exp5_results"
IDX_DIR      = "/kaggle/input/exp1-results-anubhav"   # ← your uploaded dataset
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Setup done.")

Setup done.


In [2]:
# ── Cell 2: Data Loading ──────────────────────────────────────
def flatten_dataset(ds):
    loader = DataLoader(ds, batch_size=512, shuffle=False, num_workers=2)
    Xs, ys = [], []
    for x, y in loader:
        Xs.append(x.view(x.size(0), -1).numpy())
        ys.append(y.numpy())
    return np.concatenate(Xs), np.concatenate(ys)

def get_cifar_flat(dataset_name):
    if dataset_name == "cifar10":
        mean, std = (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)
        cls = torchvision.datasets.CIFAR10
    else:
        mean, std = (0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)
        cls = torchvision.datasets.CIFAR100
    T = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
    train_ds = cls(root="/kaggle/working", train=True,  download=True, transform=T)
    test_ds  = cls(root="/kaggle/working", train=False, download=True, transform=T)
    print(f"Flattening {dataset_name} train...")
    X_train, y_train = flatten_dataset(train_ds)
    print(f"Flattening {dataset_name} test...")
    X_test,  y_test  = flatten_dataset(test_ds)
    return X_train, y_train, X_test, y_test

In [3]:
# ── Cell 3: PCA-100 Features ──────────────────────────────────
def get_pca_features(X_train, X_test, seed, n_components=100):
    # Normalize to [0,1] instead of StandardScaler — much faster on large arrays
    X_tr = X_train / 255.0 if X_train.max() > 1.0 else X_train
    X_te = X_test  / 255.0 if X_test.max()  > 1.0 else X_test
    pca    = PCA(n_components=n_components, random_state=seed, svd_solver="randomized")
    X_tr_p = pca.fit_transform(X_tr)
    X_te_p = pca.transform(X_te)
    return X_tr_p, X_te_p

In [ ]:
# ── Cell 4: Main EXP-5 Loop (uses saved coreset indices) ──────
SELECTORS = ["random", "margin", "kcenter", "gaussian_margin"]

def run_exp5(dataset_name):
    print(f"\n{'='*60}\nEXP-5 · {dataset_name.upper()}\n{'='*60}")

    X_train, y_train, X_test, y_test = get_cifar_flat(dataset_name)

    # Compute PCA once for the dataset (not per seed)
    print("Computing PCA-100...")
    X_tr_pca, X_te_pca = get_pca_features(X_train, X_test, seed=42)
    del X_train, X_test  # free RAM immediately
    print("PCA done.")

    all_results = []

    for seed in SEEDS:
        print(f"\n── Seed {seed} ──")

        for sel in SELECTORS:
            idx_file = f"{IDX_DIR}/exp1_{dataset_name}_{sel}_seed{seed}_idx.npy"
            if not os.path.exists(idx_file):
                print(f"  MISSING: {idx_file} — skipping")
                continue
            idx = np.load(idx_file)

            X_core = X_tr_pca[idx]
            y_core = y_train[idx]
            print(f"  Selector: {sel}  |  coreset size: {len(idx)}")

            for model_name, clf in [
                ("SVM", SVC(kernel="rbf", C=10, gamma="scale", random_state=seed, cache_size=2000)),
                ("RF",  RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)),
            ]:
                t0 = time.time()
                clf.fit(X_core, y_core)
                preds = clf.predict(X_te_pca)
                acc = accuracy_score(y_test, preds) * 100
                f1  = f1_score(y_test, preds, average="macro") * 100
                print(f"    {model_name}: Acc={acc:.2f}%  F1={f1:.2f}%  ({time.time()-t0:.1f}s)")

                all_results.append({
                    "dataset": dataset_name, "seed": seed,
                    "selector": sel, "model": model_name,
                    "acc": round(acc, 4), "macro_f1": round(f1, 4),
                })

    df = pd.DataFrame(all_results)
    df.to_csv(f"{RESULTS_DIR}/exp5_raw_{dataset_name}.csv", index=False)
    return df

df_c10  = run_exp5("cifar10")
df_c100 = run_exp5("cifar100")


EXP-5 · CIFAR10


100%|██████████| 170M/170M [00:02<00:00, 76.7MB/s] 


Flattening cifar10 train...
Flattening cifar10 test...
Computing PCA-100...


In [ ]:
# ── Cell 5: Summary Table (mean ± std over 3 seeds) ───────────
for ds_name, df in [("cifar10", df_c10), ("cifar100", df_c100)]:
    print(f"\n── {ds_name.upper()}: Acc@1 mean ± std ──")
    summary = (
        df.groupby(["selector", "model"])["acc"]
          .agg(mean=("acc", "mean"), std=("acc", "std"))
          .round(2)
    )
    summary["mean ± std"] = summary["mean"].astype(str) + " ± " + summary["std"].astype(str)
    print(summary[["mean ± std"]].to_string())
    summary.to_csv(f"{RESULTS_DIR}/exp5_summary_{ds_name}.csv")

In [ ]:
# ── Cell 6: Comparison Table (SVM vs RF vs MLP) ───────────────
# Load your EXP-1 MLP summary
mlp_df = pd.read_csv(f"{IDX_DIR}/exp1_summary.csv")
print("EXP-1 summary columns:", mlp_df.columns.tolist())
print(mlp_df.head())
# Inspect what columns you have, then adjust the merge below

# After inspecting, build the 3-way comparison for CIFAR-10:
# Expected: exp1_summary has columns like selector, dataset, acc_mean or similar
# Adjust column names to match what you actually see above

In [ ]:
# ── Cell 7: Ranking Consistency Check ─────────────────────────
for ds_name, df in [("cifar10", df_c10), ("cifar100", df_c100)]:
    print(f"\nSelector ranking — {ds_name.upper()}")
    rank_tbl = (
        df.groupby(["selector", "model"])["acc"]
          .mean()
          .unstack("model")
          .rank(ascending=False)
          .astype(int)
    )
    print(rank_tbl.to_string())
    rank_tbl.to_csv(f"{RESULTS_DIR}/exp5_ranking_{ds_name}.csv")

print(f"\nAll done. Results saved to {RESULTS_DIR}")